In [ ]:
!pip install crewai

In [ ]:
!pip install langchain
!pip install langchain-groq
!pip install python-dotenv
!pip install groq
!pip install litellm

In [ ]:
%%writefile .env
GROQ_API_KEY="Enter your api key here"


Overwriting .env


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
print(f"GROQ_API_KEY: {groq_api_key}")

In [ ]:
from crewai import Agent
from langchain_groq import ChatGroq

In [ ]:
!pip install agents

In [ ]:
llm = ChatGroq(
    model = "groq/llama-3.1-8b-instant",
    temperature = 0.3
)

logistic_research = Agent(
    role = "Logistic Research Specialist",
    goal = "Research and find out accurate and relevant information about the logistic topic provided",
    backstory = "Expert researcher who is capable of gathering relevant, precise and accurate information about any logistics topic",
    llm = "groq/llama-3.1-8b-instant",
    verbose = True
)
logistic_analysis = Agent(
    role = "Logistic Data Analyst",
    goal = "Analyze the logistics research data, identify key trends, patterns, and insights, and summarize findings in a clear and actionable manner.",
    backstory = "A meticulous logistics data analyst with a proven track record of extracting meaningful insights from complex datasets, transforming raw information into strategic recommendations.",
    llm = "groq/llama-3.1-8b-instant",
    verbose = True
)
logistic_summarizer = Agent(
    role = "Teacher",
    goal = "Explain the findings clearly for students",
    backstory = "An eperienced educator who simplifies complex ideas.",
    llm = "groq/llama-3.1-8b-instant",
    verbose = True
)

In [ ]:
from crewai import Task

research_task = Task(
    description = "Research about:{topic}. Collect important facts and explanations about the topic.",
    expected_output="Research data about the topic with some key facts as bullet points and other explanations",
    agent = logistic_research
)
analysis_task = Task(
    description = "Analyse the research findings and identify the key facts and information.",
    expected_output="A list of important insights derived from the research agent's results",
    agent = logistic_analysis
)
summary_task = Task(
    description = "Summarize the results for research agent and analysis agent and explain the insights in simple words for students.",
    expected_output="Simplified summary of the conclusions and works of research agent and analysis agent",
    agent = logistic_summarizer
)

In [ ]:
!pip install tasks
from crewai import Crew

In [ ]:
crew = Crew(
    agents = [logistic_research, logistic_analysis, logistic_summarizer],
    tasks = [research_task, analysis_task, summary_task],
    verbose = True
)

In [ ]:
"""
topic = input("Enter your Query: ")
result = crew.kickoff(
    inputs={"topic":topic}
)
print("\n Final Result: \n")
print(result)
"""

'\ntopic = input("Enter your Query: ")\nresult = crew.kickoff(\n    inputs={"topic":topic}\n)\nprint("\n Final Result: \n")\nprint(result)\n'

In [ ]:
!pip install streamlit pyngrok -q

In [ ]:
%%writefile app.py

import streamlit as st
from crewai import Agent, Task, Crew
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

st.set_page_config(page_title="CrewAI Logistic Research Assistant", layout="wide")

st.title("CrewAI Logistic Research Assistant")
st.markdown("--- Say Hello to your AI-powered Logistic Research Team ---")

# Initialize the LLM (Note: this 'llm' object is not directly used by agents in this revised setup, but kept for reference)
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.3,
    groq_api_key=groq_api_key
)

# Define Agents
logistic_research = Agent(
    role="Logistic Research Specialist",
    goal="Research and find out accurate and relevant information about the logistic topic provided",
    backstory="Expert researcher who is capable of gathering relevant, precise and accurate information about any logistics topic",
    llm="groq/llama-3.1-8b-instant", # Changed to string model name
    verbose=True,
    allow_delegation=False
)

logistic_analysis = Agent(
    role="Logistic Data Analyst",
    goal="Analyze the logistics research data, identify key trends, patterns, and insights, and summarize findings in a clear and actionable manner.",
    backstory="A meticulous logistics data analyst with a proven track record of extracting meaningful insights from complex datasets, transforming raw information into strategic recommendations.",
    llm="groq/llama-3.1-8b-instant", # Changed to string model name
    verbose=True,
    allow_delegation=False
)

logistic_summarizer = Agent(
    role="Teacher",
    goal="Explain the findings clearly for students",
    backstory="An experienced educator who simplifies complex ideas.",
    llm="groq/llama-3.1-8b-instant", # Changed to string model name
    verbose=True,
    allow_delegation=False
)

# Streamlit Input
topic = st.text_input("Enter your Query (e.g., 'export logistics')", "")

if st.button("Run CrewAI"): # Changed from st.form to st.button with st.text_input
    if topic:
        with st.spinner("Running CrewAI agents..."): # Changed to st.spinner
            # Define Tasks
            research_task = Task(
                description=f"Research about: {topic}. Collect important facts and explanations about the topic.",
                expected_output="Research data about the topic with some key facts as bullet points and other explanations",
                agent=logistic_research
            )

            analysis_task = Task(
                description="Analyse the research findings and identify the key facts and information.",
                expected_output="A list of important insights derived from the research agent's results",
                agent=logistic_analysis
            )

            summary_task = Task(
                description="Summarize the results for research agent and analysis agent and explain the insights in simple words for students.",
                expected_output="Simplified summary of the conclusions and works of research agent and analysis agent",
                agent=logistic_summarizer
            )

            # Instantiate Crew
            project_crew = Crew(
                agents=[logistic_research, logistic_analysis, logistic_summarizer],
                tasks=[research_task, analysis_task, summary_task],
                verbose=True
            )

            # Kickoff the crew
            try:
                result = project_crew.kickoff(inputs={"topic": topic})
                st.subheader("Final Result:")
                st.write(result.raw) # Displaying only the 'raw' attribute for cleaner output
            except Exception as e:
                st.error(f"An error occurred during CrewAI execution: {e}")
    else:
        st.warning("Please enter a topic to run the CrewAI.")

Overwriting app.py


In [ ]:
from pyngrok import ngrok

# Terminate any previous ngrok tunnels
ngrok.kill()

# IMPORTANT: Replace "YOUR_NGROK_AUTH_TOKEN" with your actual ngrok authtoken
# You can get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("39jo3S1OGLkctObZoVjaRPF5G3p_3fd3j23QVwmh65VvKmrsZ")

# Start a ngrok tunnel to the Streamlit app on port 8501
public_url = ngrok.connect(8501)
print(f"Streamlit App URL: {public_url}")

# Run the Streamlit app in the background
!streamlit run app.py &>/dev/null&

Streamlit App URL: NgrokTunnel: "https://nita-multiarticular-shakira.ngrok-free.dev" -> "http://localhost:8501"
